# Résolveur Public — Exploration & Pipeline

Agent de resolution de questions administratives : face a une nouvelle question,
decider s'il s'agit d'une **question deja repondue officiellement** (renvoyer la
reponse) ou d'un **cas inedit** (escalade vers un agent humain).

**Stack :** BGE-M3 (embeddings) · FAISS (index vectoriel) · CrossEncoder (reranking) ·
seuil de decision **calibre empiriquement**.

**Donnees :** sous-ensemble "Question-reponse" de `AgentPublic/service-public`
(Hugging Face), contenu officiel de service-public.fr, licence Etalab-2.0.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

from src import config
from src.data import load_dataset, train_test_split_with_novelty

## 1. Chargement

Prerequis : `python ../scripts/build_dataset.py` (telecharge et normalise les donnees).

In [ ]:
df = load_dataset()
print(f'{len(df)} questions, {df[config.COL_CATEGORY].nunique()} themes')
df.head()

## 2. Exploration

In [ ]:
theme_counts = df[config.COL_CATEGORY].value_counts()
fig, ax = plt.subplots(figsize=(9, 5))
theme_counts.plot(kind='barh', ax=ax, color='#4C72B0')
ax.invert_yaxis()
ax.set_title('Distribution des questions par theme administratif')
plt.tight_layout(); plt.show()

In [ ]:
df['q_len'] = df[config.COL_ISSUE].str.split().apply(len)
df['a_len'] = df[config.COL_RESPONSE].str.split().apply(len)
print(df[['q_len', 'a_len']].describe())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df['q_len'], bins=30, ax=axes[0], color='#55A868')
axes[0].set_title('Longueur des questions (mots)')
sns.histplot(df['a_len'], bins=30, ax=axes[1], color='#C44E52')
axes[1].set_title('Longueur des reponses (mots)')
plt.tight_layout(); plt.show()

## 3. Split train/test avec simulation de nouveaute

Le theme `config.NOVEL_CATEGORY` ("Ressources humaines") est entierement retire du
train pour simuler de vraies questions inedites.

In [ ]:
train_df, test_df = train_test_split_with_novelty(df)
print(f'train={len(train_df)}  test={len(test_df)}')
print(f"  test connus  : {(~test_df['is_novel']).sum()}")
print(f"  test nouveaux : {test_df['is_novel'].sum()} (theme '{config.NOVEL_CATEGORY}')")
assert config.NOVEL_CATEGORY not in train_df[config.COL_CATEGORY].unique()

## 4. Indexation BGE-M3 + FAISS

In [ ]:
from src.indexing import build_index, get_embedder

embedder = get_embedder()
index, meta = build_index(train_df, embedder)
print('Vecteurs indexes:', index.ntotal)

## 5. Pipeline de requete (demonstration)

In [ ]:
from src.query import IncidentResolver

resolver = IncidentResolver()
cands = resolver.retrieve_and_rerank('Comment faire une demande de logement social ?')
for c in cands:
    print(f'[{c.category}] rerank={c.rerank_score:.2f} cos={c.faiss_score:.2f} :: {c.issue}')

## 6. Calibration du seuil + evaluation

Le seuil de decision (connu vs nouveau) maximise le **F1 macro** entre les deux classes.

In [ ]:
from src.evaluation import score_test_set, calibrate_threshold, evaluate, write_report

scored = score_test_set(resolver, test_df)
best_threshold, sweep = calibrate_threshold(scored)
print('Seuil calibre:', round(best_threshold, 3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sw = sweep.sort_values('threshold')
ax.plot(sw['threshold'], sw['f1_macro'], label='F1 macro', lw=2)
ax.plot(sw['threshold'], sw['f1_known'], '--', label='F1 connu', alpha=0.7)
ax.plot(sw['threshold'], sw['f1_novel'], '--', label='F1 nouveau', alpha=0.7)
ax.axvline(best_threshold, color='red', ls=':', label=f'seuil optimal={best_threshold:.2f}')
ax.set_xlabel('seuil'); ax.set_ylabel('F1'); ax.legend()
ax.set_title('Calibration du seuil de decision'); plt.tight_layout(); plt.show()

In [ ]:
metrics = evaluate(scored, best_threshold)
write_report(metrics, sweep)

print(f"F1 macro   : {metrics['macro_f1']:.3f}")
print(f"F1 connu   : {metrics['per_class']['known']['f1']:.3f}")
print(f"F1 nouveau : {metrics['per_class']['novel']['f1']:.3f}")
print(f"Recall@{metrics['k']}    : {metrics['recall_at_k']:.3f}")
print('Matrice de confusion:', metrics['confusion'])